In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import json
from datetime import datetime
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Loading Data

In [84]:
# files = ["preprocessed_data_1.json", "preprocessed_data_2.json", "preprocessed_data_3.json"]
files = ["preprocessed_data_modified.json"]
data = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        data += [pd.DataFrame(json.load(f))]

df = pd.concat(data, ignore_index=True)
print(df.head())
print(len(df))

             start station              end station      hour  week day  \
0          Bến xe An Sương                  Tân Sơn  4.650000         6   
1                  Tân Sơn    Ngã tư Thoại Ngọc Hầu  5.200000         6   
2    Ngã tư Thoại Ngọc Hầu  Công ty Bông Bạch Tuyết  5.333333         6   
3  Công ty Bông Bạch Tuyết       Chợ Trần Văn Quang  5.350000         6   
4       Chợ Trần Văn Quang            Chợ Tân Phước  5.366667         6   

   distance (m)  duration (s)  
0   3928.179591          1966  
1   3415.784008           452  
2    517.287167           100  
3    337.855837            54  
4    870.047606           116  
264272


# Preprocessing

In [85]:
# Con số 3000 ở đây là theo quy định pháp luật
# uyến xe buýt có các điểm dừng đón, trả khách. Khoảng cách tối đa giữa hai điểm dừng đón, 
# trả khách liền kề trong nội thành, nội thị là 700 mét, ngoại thành, ngoại thị là 3.000 mét.
# 3000 ở đây là worst case
df = df[(df["distance (m)"] <= 3000) & (df["duration (s)"] <= 1800)]
print(len(df))

223461


In [86]:
df["route"] = df["start station"] + "_" + df["end station"]
route_mean = (
    df.groupby("route")["duration (s)"]
      .mean()
      .rename("route_avg_duration")
)
df = df.merge(route_mean, on="route")

df["weekend"] = (df["week day"]>=5).astype(int)
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["delay_from_avg"] = df["duration (s)"] - df["route_avg_duration"]

df = df.drop(columns=["week day", "hour", "route"])
# df = df.drop(columns=["start station"])
# df = df.drop(columns=["end station"])

print(df.head(5))
print(len(df))

df.to_json("dataset.json", orient="records", force_ascii=False, indent=4)

             start station              end station  distance (m)  \
0    Ngã tư Thoại Ngọc Hầu  Công ty Bông Bạch Tuyết    517.287167   
1  Công ty Bông Bạch Tuyết       Chợ Trần Văn Quang    337.855837   
2       Chợ Trần Văn Quang            Chợ Tân Phước    870.047606   
3            Chợ Tân Phước    Vòng xoay Lê Đại Hành    315.499277   
4    Vòng xoay Lê Đại Hành                  Parkson    872.258094   

   duration (s)  route_avg_duration  weekend  hour_sin  hour_cos  \
0           100          122.962617        1  0.984808  0.173648   
1            54           64.764706        1  0.985556  0.169350   
2           116          182.700000        1  0.986286  0.165048   
3            50           65.357143        1  0.987688  0.156434   
4           110          146.333333        1  0.988362  0.152123   

   delay_from_avg  
0      -22.962617  
1      -10.764706  
2      -66.700000  
3      -15.357143  
4      -36.333333  
223461


In [108]:
X = df.drop(columns=["duration (s)"])
Y = df["duration (s)"]
# X = df.drop(columns=["delay_from_avg"])
# Y = df["delay_from_avg"]


# categorical_cols = ["route"]
categorical_cols = ["start station", "end station"]
numeric_cols = ["distance (m)", "weekend", "hour_sin", "hour_cos", "route_avg_duration"]
# numeric_cols = ["distance (m)", "week day", "hour", "weekend"]

lr_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", 'passthrough', numeric_cols)
    ]
)

gb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

In [88]:
linear_model = Pipeline([
    ("preprocess", lr_preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, Y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [89]:
pred_lr = linear_model.predict(X_test)
print(pred_lr)

print("MAE:", mean_absolute_error(Y_test, pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(Y_test, pred_lr)))
print("R2:", r2_score(Y_test, pred_lr))

[146.2359311   50.37871508 248.51908305 ... 110.25639307 198.81453667
 294.69347586]
MAE: 65.85055217356803
RMSE: 153.36104951426444
R2: 0.6615617200864105


In [90]:
# Tạo ra file
joblib.dump(linear_model, "linear_regression_model.pkl")

['linear_regression_model.pkl']

In [119]:
rf_model = Pipeline([
    ("preprocess", rf_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
        verbose=1
    ))
])

rf_model.fit(X_train, Y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed: 10.1min
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed: 13.5min finished


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [120]:
pred_rf = rf_model.predict(X_test)

print("MAE:", mean_absolute_error(Y_test, pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(Y_test, pred_rf)))
print("R2:", r2_score(Y_test, pred_rf))

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


MAE: 62.113841988678324
RMSE: 154.42483679901173
R2: 0.6568502891276582


[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.3s finished


In [121]:
# Tạo ra file
joblib.dump(rf_model, "randomforest_model.pkl")

['randomforest_model.pkl']

In [116]:
gb_model = Pipeline([
    ("preprocess", gb_preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        verbose=1
    ))
])

gb_model.fit(X_train, Y_train)

      Iter       Train Loss   Remaining Time 
         1       67198.9760            2.20m
         2       63136.4472            2.12m
         3       59461.8521            2.23m
         4       56140.4532            2.23m
         5       53131.6294            2.18m
         6       50410.7341            2.16m
         7       47949.3911            2.11m
         8       45719.0565            2.08m
         9       43700.0312            2.09m
        10       41872.6520            2.06m
        20       30819.5424            1.93m
        30       26592.0848            1.99m
        40       24922.2411            2.10m
        50       24218.5265            1.99m
        60       23641.9585            1.89m
        70       23289.4672            1.82m
        80       23097.7744            1.77m
        90       22922.0836            1.69m
       100       22751.0609            1.63m
       200       22060.9935           49.06s
       300       21749.7532            0.00s


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [117]:
pred_gb = gb_model.predict(X_test)

print("MAE:", mean_absolute_error(Y_test, pred_gb))
print("RMSE:", np.sqrt(mean_squared_error(Y_test, pred_gb)))
print("R2:", r2_score(Y_test, pred_gb))

MAE: 62.96200088513882
RMSE: 148.13730871445313
R2: 0.6842246421922089


In [122]:
# Tạo ra file
joblib.dump(rf_model, "gradientboosting_model.pkl")

['gradientboosting_model.pkl']